In [ ]:
# 0. Inisialisasi SparkSession & Folder Simulasi
import os
import shutil
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as _sum, count, window, avg

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

# Pengaturan partition shuffle
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.sparkContext.setLogLevel("ERROR")

# Menyiapkan folder simulasi
folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

# Bersihkan folder jika sudah ada sebelumnya
for folder in [folder_stream, folder_output, folder_checkpoint]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

print("SparkSession dan folder simulasi siap!")

In [ ]:
# A. Menyiapkan Streaming Source

# 1. Definisi skema eksplisit (StructType)
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# 2. Definisi readStream dari folder tugas10_stream
df_stream = spark.readStream \
    .schema(skema_pesanan) \
    .option("header", "true") \
    .csv(folder_stream)

In [ ]:
# B. Agregasi per Cabang

# 1. Menambahkan kolom turunan total_harga (qty * harga_satuan)
df_with_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# 2. Agregasi groupBy("nama_cabang") menghitung total total_harga dan jumlah pesanan
df_cabang_agg = df_with_total.groupBy("nama_cabang").agg(
    _sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# 3. Jalankan streaming query dengan sink memory dan queryName="agregasi_cabang"
query1 = df_cabang_agg.writeStream \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .outputMode("complete") \
    .start()

In [ ]:
# C. Agregasi Berbasis Window Waktu

# 1. Definisi ulang readStream untuk query kedua dari folder yang sama
df_stream_window = spark.readStream \
    .schema(skema_pesanan) \
    .option("header", "true") \
    .csv(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# 2. Agregasi total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan
df_window_agg = df_stream_window.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    _sum("total_harga").alias("total_pendapatan_window"),
    count("pesanan_id").alias("jumlah_pesanan_window")
)

# 3. Jalankan streaming query kedua dengan sink memory dan queryName="agregasi_waktu"
query2 = df_window_agg.writeStream \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .outputMode("complete") \
    .start()

In [ ]:
# D. Menguji dengan Data Simulasi,
# Mengirimkan 3 batch data secara bertahap dengan jeda time.sleep() dan menampilkan isi tabel agregasi_cabang

# --- BATCH 1 ---
csv_batch1 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P001,Jakarta,Kopi Kenangan,2,18000,2026-11-05 14:00:00
P002,Bandung,Es Kopi Susu,1,15000,2026-11-05 14:00:30
P003,Jakarta,Americano,3,20000,2026-11-05 14:01:00
"""

with open(f"{folder_stream}/batch1.csv", "w") as f:
    f.write(csv_batch1)

print("--- Menulis Batch 1 ---")
time.sleep(5)
spark.sql("SELECT * FROM agregasi_cabang").show()

# --- BATCH 2 ---
csv_batch2 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P004,Surabaya,Matcha Latte,2,22000,2026-11-05 14:01:15
P005,Bandung,Kopi Kenangan,2,18000,2026-11-05 14:01:45
P006,Jakarta,Es Kopi Susu,1,15000,2026-11-05 14:02:00
"""

with open(f"{folder_stream}/batch2.csv", "w") as f:
    f.write(csv_batch2)

print("--- Menulis Batch 2 ---")
time.sleep(5)
spark.sql("SELECT * FROM agregasi_cabang").show()

# --- BATCH 3 ---
csv_batch3 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P007,Surabaya,Americano,1,20000,2026-11-05 14:02:30
P008,Jakarta,Matcha Latte,2,22000,2026-11-05 14:02:50
P009,Bandung,Americano,2,20000,2026-11-05 14:03:00
"""

with open(f"{folder_stream}/batch3.csv", "w") as f:
    f.write(csv_batch3)

print("--- Menulis Batch 3 ---")
time.sleep(5)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Opsional: Tampilkan juga hasil agregasi window waktu dari query 2
print("--- Hasil Agregasi Window Waktu ---")
spark.sql("SELECT * FROM agregasi_waktu ORDER BY window").show(truncate=False)

In [ ]:
# E. Sink Berbasis Berkas (Tantangan)

# 1. Menjalankan streaming query ketiga (sink Parquet)
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# 2. Kirim satu batch data baru (Batch 4)
csv_batch4 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P010,Yogyakarta,Kopi Kenangan,3,18000,2026-11-05 14:03:15
P011,Yogyakarta,Es Kopi Susu,2,15000,2026-11-05 14:03:30
"""

with open(f"{folder_stream}/batch4.csv", "w") as f:
    f.write(csv_batch4)

print("--- Menulis Batch 4 untuk Sink Parquet ---")
time.sleep(6)

# 3. Verifikasi berkas Parquet tersimpan dan baca kembali sebagai DataFrame batch biasa
print("Isi folder tugas10_output:")
os.system("ls tugas10_output")

print("\nMembaca kembali data dari folder Parquet:")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

In [ ]:
# Pembersihan Resource / Menghentikan Stream
# Di sel paling akhir notebook Anda, jalankan pembersihan query dan sesi Spark:

# Menghentikan seluruh query streaming yang aktif
query1.stop()
query2.stop()
query3.stop()

# Menghentikan SparkSession
spark.stop()
print("Seluruh query dan SparkSession berhasil dihentikan.")